# Correlation Tutorial 5: Plotting, Styling, and Publication Vector Export

Welcome to **Correlation** Tutorial 5. This guide demonstrates how to generate publication-quality figures, apply colorblind-safe scientific styling (Okabe-Ito palette), compare multiple simulation states, and export in vector formats (**PDF**, **SVG**) and high-resolution **PNG**.

### Key Topics Covered
1. Generating atomic structures with controlled thermal disorder.
2. Computing high-resolution **Radial Distribution Functions (RDF)** with `correlation`.
3. Designing publication-grade multi-panel figures with Matplotlib.
4. Customizing line markers, legends, LaTeX notation, and inset zooms.
5. Exporting vector graphics for journals and presentations.


## 1. Imports and Setup


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import correlation
from pathlib import Path

# Create export directory for output figures
export_dir = Path("figures_export")
export_dir.mkdir(exist_ok=True)

print("Correlation version:", getattr(correlation, "__version__", "4.0.0"))


## 2. Generating Comparative Structures (Ideal vs Thermally Disordered)

We construct an FCC Copper lattice ($a = 3.615$ Å) replicated $4\times4\times4$, and compare the ideal lattice with configurations subjected to Gaussian thermal fluctuations $\sigma = 0.05$ Å and $\sigma = 0.15$ Å.


In [ ]:
a = 3.615
n_rep = 4
L = a * n_rep

# Base FCC fractional coordinates
fcc_basis = np.array([
    [0.0, 0.0, 0.0],
    [0.5, 0.5, 0.0],
    [0.5, 0.0, 0.5],
    [0.0, 0.5, 0.5]
])

# Replicate into supercell
positions = []
for ix in range(n_rep):
    for iy in range(n_rep):
        for iz in range(n_rep):
            offset = np.array([ix, iy, iz])
            for b in fcc_basis:
                positions.append((b + offset) * a)

ideal_pos = np.array(positions)
symbols = ["Cu"] * len(ideal_pos)

# Disordered configurations
np.random.seed(42)
pos_warm = ideal_pos + np.random.normal(0, 0.05, ideal_pos.shape)
pos_hot = ideal_pos + np.random.normal(0, 0.15, ideal_pos.shape)

# Build Correlation Cells using bulk zero-copy from_arrays
cell_ideal = correlation.Cell([L, 0.0, 0.0], [0.0, L, 0.0], [0.0, 0.0, L])
cell_ideal.from_arrays(ideal_pos, symbols)

cell_warm = correlation.Cell([L, 0.0, 0.0], [0.0, L, 0.0], [0.0, 0.0, L])
cell_warm.from_arrays(pos_warm, symbols)

cell_hot = correlation.Cell([L, 0.0, 0.0], [0.0, L, 0.0], [0.0, 0.0, L])
cell_hot.from_arrays(pos_hot, symbols)

print(f"Created 3 cells with {len(ideal_pos)} atoms each in {L:.2f} Å box.")


## 3. High-Performance Distribution Analysis

Calculate RDF $g(r)$ up to $r_{\max} = 8.0$ Å with bin width $\Delta r = 0.02$ Å.


In [ ]:
r_max = 8.0
bin_width = 0.02

curves = {}
for label, cell in [("Ideal FCC (0 K)", cell_ideal), ("Warm (σ = 0.05 Å)", cell_warm), ("Hot (σ = 0.15 Å)", cell_hot)]:
    df = correlation.DistributionFunctions.from_cell(cell, 0.0)
    df.calculate_rdf(r_max, bin_width)
    hist = df.get_histogram("RDF")
    curves[label] = (hist.bins, hist.partials["Total"])

print("RDF calculations complete.")


## 4. Publication-Quality Multi-Curve Plotting

Apply Okabe-Ito colorblind palette, crisp typography, and an inset detailing the first coordination shell.


In [ ]:
# Set publication styling parameters
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Helvetica']
mpl.rcParams['axes.linewidth'] = 1.2
mpl.rcParams['xtick.direction'] = 'in'
mpl.rcParams['ytick.direction'] = 'in'

palette = ["#0072B2", "#D55E00", "#009E73"]

fig, ax = plt.subplots(figsize=(7, 4.5), dpi=300)

for idx, (label, (r, g_r)) in enumerate(curves.items()):
    ax.plot(r, g_r, label=label, color=palette[idx], lw=1.8, alpha=0.9)

ax.set_xlabel(r"Distance $r$ (Å)", fontsize=12)
ax.set_ylabel(r"Radial Distribution $g(r)$", fontsize=12)
ax.set_xlim(1.5, 7.5)
ax.set_ylim(0, None)
ax.grid(True, linestyle=":", alpha=0.4, color="#888888")
ax.legend(frameon=True, facecolor="white", edgecolor="#cccccc", fontsize=10, loc="upper right")

# Inset: First peak detail
ax_ins = ax.inset_axes([0.18, 0.45, 0.35, 0.45])
for idx, (label, (r, g_r)) in enumerate(curves.items()):
    ax_ins.plot(r, g_r, color=palette[idx], lw=1.8)
ax_ins.set_xlim(2.2, 2.9)
ax_ins.set_ylim(0, 15)
ax_ins.set_title(r"1$^{\mathrm{st}}$ Neighbor Shell", fontsize=9)
ax_ins.grid(True, linestyle=":", alpha=0.3)
ax.indicate_inset_zoom(ax_ins, edgecolor="black", alpha=0.5)

plt.tight_layout()
plt.show()


## 5. Multi-Format Vector Export

Export figures across multiple formats suited for journal submissions, LaTeX compilation, and web display.


In [ ]:
# Export Vector and Raster formats
pdf_path = export_dir / "rdf_comparison.pdf"
svg_path = export_dir / "rdf_comparison.svg"
png_path = export_dir / "rdf_comparison.png"

fig.savefig(pdf_path, format="pdf", bbox_inches="tight")
fig.savefig(svg_path, format="svg", bbox_inches="tight")
fig.savefig(png_path, format="png", dpi=300, bbox_inches="tight")

print(f"Saved PDF: {pdf_path} ({pdf_path.stat().st_size / 1024:.1f} KB)")
print(f"Saved SVG: {svg_path} ({svg_path.stat().st_size / 1024:.1f} KB)")
print(f"Saved PNG: {png_path} ({png_path.stat().st_size / 1024:.1f} KB)")


## 6. Summary
- Used `Cell.from_arrays` for fast atomic configuration loading.
- Computed high-resolution RDF with `correlation.DistributionFunctions`.
- Rendered publication-ready plots with Okabe-Ito colorblind palette and insets.
- Exported vector assets in PDF and SVG formats ready for LaTeX manuscript integration.
